# Portfolio Optimisation: worked example

An end-to-end walkthrough of the toolkit: load data, run econometric
diagnostics, calibrate stochastic processes, build hierarchical
allocations, denoise the covariance, blend Bayesian views, simulate the
risk distribution, and report performance. Every stochastic step is
seeded, so re-running this notebook reproduces the figures and tables
exactly.

## 1. Configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from rich.console import Console

from portfolio_optimisation.config import Settings

TICKERS = ["IYW", "VGT", "IYF", "IYR", "XLU", "IYK"]
START_DATE = "2017-01-01"
SEED = 12345
ANNUAL = 252

settings = Settings(seed=SEED)
console = Console()
output_dir = Path("outputs") / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 4)

## 2. Market data

Loaded through the repository, which serves the validated parquet snapshot.

In [ ]:
from portfolio_optimisation.infra.repositories import YfinanceParquetRepository

repo = YfinanceParquetRepository(cache_path=settings.data_cache_path)
prices, returns = repo.load_prices(TICKERS, START_DATE)
print(f"{prices.shape[0]} observations, {prices.shape[1]} assets")
prices.tail()

## 3. Econometric diagnostics

Normality, stationarity, autocorrelation, heteroskedasticity, ARCH effects and structural breaks.

In [ ]:
from portfolio_optimisation.econometrics import Econometrics

Econometrics(returns).run_all_tests()

## 4. Stochastic-process calibration

Maximum-likelihood fits of Geometric Brownian Motion and Ornstein-Uhlenbeck dynamics.

In [ ]:
from portfolio_optimisation.sde import SDEFitter

fitter = SDEFitter(prices)
gbm_params = fitter.fit_gbm()
ou_params = fitter.fit_ou()
gbm_params

## 5. Hierarchical allocation

HRP, HERC and NCO weights compared side by side.

In [ ]:
from portfolio_optimisation.optim import HRPModel, herc_weights, nco_weights

hrp = HRPModel(returns)
hrp.optimize(linkage_method=settings.linkage_method)
hrp_weights = hrp.clean_weights()

allocations = pd.DataFrame(
    {
        "HRP": hrp_weights,
        "HERC": herc_weights(returns),
        "NCO": nco_weights(returns),
    }
)
allocations

## 6. Denoising and Black-Litterman

Marchenko-Pastur denoising of the covariance and a Bayesian blend against the HRP equilibrium prior.

In [ ]:
from portfolio_optimisation.optim import black_litterman_weights, denoise_covariance

q = returns.shape[0] / returns.shape[1]
covariance = returns.cov()
denoised_covariance = denoise_covariance(covariance, q=q, detone=True)
posterior = black_litterman_weights(covariance, hrp_weights)
posterior.weights

## 7. Risk analysis

Student t-copula simulation, Value-at-Risk and Conditional VaR, plus coherent risk measures.

In [ ]:
from portfolio_optimisation.risk import (
    CopulaRiskAnalyser,
    calculate_performance_metrics,
    calculate_risk_metrics,
    entropic_value_at_risk,
    exponential_spectrum,
    spectral_risk_measure,
    wang_transform_risk,
)

analyser = CopulaRiskAnalyser(returns, hrp_weights)
analyser.fit_marginal_distributions()
analyser.fit_copula()
simulated = analyser.run_simulation(n_simulations=20000, seed=SEED)

risk = calculate_risk_metrics(
    simulated, alpha=settings.var_alpha, method=settings.var_method
)
portfolio_returns = returns.dot(hrp_weights.reindex(returns.columns).fillna(0.0))
performance = calculate_performance_metrics(
    portfolio_returns, risk_free_rate=settings.risk_free_rate
)

coherent = {
    "EVaR (5%)": entropic_value_at_risk(portfolio_returns, alpha=0.05),
    "Spectral (exp)": spectral_risk_measure(
        portfolio_returns, exponential_spectrum(0.05)
    ),
    "Wang (lambda=0.25)": wang_transform_risk(portfolio_returns, 0.25),
}
pd.Series({**risk, **coherent})

## 8. Sharpe diagnostics

Probabilistic and Deflated Sharpe Ratios with a stationary-bootstrap confidence interval.

In [ ]:
from portfolio_optimisation.risk import (
    deflated_sharpe_ratio,
    probabilistic_sharpe_ratio,
    stationary_bootstrap_sharpe_ci,
)

daily_rf = settings.risk_free_rate / ANNUAL
psr = probabilistic_sharpe_ratio(portfolio_returns, risk_free_rate=daily_rf)
dsr = deflated_sharpe_ratio(portfolio_returns, n_trials=10, risk_free_rate=daily_rf)
lower, upper, _ = stationary_bootstrap_sharpe_ci(
    portfolio_returns, n_resamples=1000, seed=SEED
)
print(f"PSR = {psr:.3f}")
print(f"DSR (10 trials) = {dsr:.3f}")
print(f"Bootstrap 95% Sharpe CI = ({lower:.3f}, {upper:.3f})")

## 9. Figures

Publication-grade figures, saved under `outputs/figures/`.

In [ ]:
from portfolio_optimisation.viz import PortfolioVisualiser

mu = returns.mean() * ANNUAL
annual_cov = returns.cov() * ANNUAL
weight_map = {
    "HRP": hrp_weights,
    "HERC": allocations["HERC"],
    "NCO": allocations["NCO"],
}
viz = PortfolioVisualiser(mu, annual_cov, weight_map)
viz.plot_efficient_frontier(save_path=output_dir / "efficient_frontier.png")
viz.plot_comparative_weights(save_path=output_dir / "weights.png")
viz.plot_dendrogram(hrp, save_path=output_dir / "dendrogram.png")
viz.plot_correlation_matrix(hrp, ordered=True, save_path=output_dir / "correlation.png")
plt.show()

## 10. Report

Consolidated performance and risk table.

In [ ]:
from portfolio_optimisation.infra import generate_final_report

generate_final_report(console, {"HRP": {**performance, **risk}})